# Bulk Address Lookup

**Goal:** Retrieve forecasts for a list of addresses and export the results to CSV.

This is the most common starting point for analysts who have a portfolio, watchlist, or acquisition pipeline they want to screen against Homecastr forecasts.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/homecastr/homecastr-cookbook/blob/main/examples/getting_started/02_bulk_lookup.ipynb)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from homecastr import HomecastrClient

client = HomecastrClient(os.getenv("HOMECASTR_API_KEY", "hc_demo_public_readonly"))

In [ ]:
# Replace with your own address list or load from a CSV:
# addresses = pd.read_csv("my_portfolio.csv")["address"].tolist()

addresses = [
    "4521 Westheimer Rd Houston TX 77027",
    "1234 S Congress Ave Austin TX 78704",
    "2800 N Milwaukee Ave Chicago IL 60618",
    "567 Valencia St San Francisco CA 94110",
    "1045 Ocean Ave Brooklyn NY 11230",
    "3401 N Williams Ave Portland OR 97227",
    "512 NW 84th St Seattle WA 98117",
    "2100 NW 2nd Ave Miami FL 33127",
]

print(f"Fetching forecasts for {len(addresses)} addresses...")

In [ ]:
# retrieve_many handles errors gracefully and returns a DataFrame
df = client.forecast.by_address.retrieve_many(addresses, year=2028)

print(df.shape)
df.head()

In [ ]:
# Add derived columns
df["upside"] = (df["p90"] - df["p50"]) / df["p50"] * 100
df["downside"] = (df["p50"] - df["p10"]) / df["p50"] * 100
df["risk_adjusted_growth"] = df["appreciation_pct"] * df["reliability"]

display_cols = ["address", "current_value", "p50", "appreciation_pct",
                "upside", "downside", "reliability", "risk_adjusted_growth"]

print(df[display_cols].sort_values("risk_adjusted_growth", ascending=False).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: appreciation by market
df_sorted = df.sort_values("appreciation_pct", ascending=True)
short_labels = [a.split(" ")[-2] + " " + a.split(" ")[-1] for a in df_sorted["address"]]
axes[0].barh(short_labels, df_sorted["appreciation_pct"], color="steelblue", alpha=0.8)
axes[0].set_xlabel("5-Year Appreciation (%)")
axes[0].set_title("Forecast Appreciation by Market")

# Right: reliability vs appreciation scatter
sc = axes[1].scatter(df["reliability"], df["appreciation_pct"],
                     s=df["current_value"] / 5000, c=df["risk_adjusted_growth"],
                     cmap="RdYlGn", alpha=0.8, edgecolors="k", linewidths=0.5)
plt.colorbar(sc, ax=axes[1], label="Risk-adjusted growth")
axes[1].set_xlabel("Reliability Score")
axes[1].set_ylabel("Appreciation (%)")
axes[1].set_title("Reliability vs. Appreciation\n(bubble size = current value)")

plt.tight_layout()
plt.show()

In [ ]:
# Export to CSV
output_path = "homecastr_forecasts.csv"
df.to_csv(output_path, index=False)
print(f"Saved {len(df)} rows to {output_path}")